# 03 · Blind judge file and final score
No GPU needed.
1. Build `judge_items.csv` (both systems mixed, shuffled, system names hidden, 10% hidden duplicates).
2. Send **only** `judge_items.csv` to the judge. Keep `judge_key.csv` private.
3. Put the returned `judge_scores.csv` in the same folder and run Step 2. It writes `report.md`, `report.json` and `results.png`.

## Setup

In [ ]:
# ---- Settings (edit once) ----
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
LOCAL = "/content/data"            # local copy of the dataset (Drive I/O is slow)

In [ ]:
# ---- Mount Drive ----
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
!pip install -q sacrebleu==2.5.1
import os, json
import numpy as np, pandas as pd

In [ ]:
# ---- Copy the frozen dataset (v1) to local disk and check it is unchanged ----
import hashlib, shutil
DATA_FILES = {   # sha256 from data/v1/MANIFEST.json (frozen 2026-10-21)
    "splits/train.csv": "9fa401ea9c8680df38b1255834420c7f60e0f7b5471410fa69e6ef981c563b63",
    "splits/test.csv": "6f374a18735e57741c73a8f1292bc3a441d23142dac82306247cb373d45a327a",
    "structured/structured.csv": "95dc97809eac2722db1c4d2aaea2865a44d9eabc5b229a30c932f06414b60258",
}
for rel, sha in DATA_FILES.items():
    dst = f"{LOCAL}/{rel}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f"{DRIVE_ROOT}/data/v1/{rel}", dst)
    got = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    assert got == sha, f"STOP: {rel} differs from the frozen v1 file"
print("dataset OK:", ", ".join(DATA_FILES))

## Code

In [ ]:
# ---- Chart style: categorical slots 1/2 for the two systems ----
C_BASE, C_FT = "#2a78d6", "#eb6834"
C_TEXT, C_MUTED, C_GRID = "#0b0b0b", "#52514e", "#e4e3df"


def style_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(C_MUTED)
    ax.tick_params(colors=C_MUTED, labelsize=9)
    ax.grid(axis="y", color=C_GRID, linewidth=0.8)
    ax.set_axisbelow(True)

In [ ]:
# ---- Build one blind judge file from several systems' predictions ----
import os
import numpy as np
import pandas as pd


def make_judge_file(preds, out_dir, split="test", dup_frac=0.10, seed=42, data_dir=LOCAL):
    """preds: {"baseline": path, "finetuned": path}.
    Writes judge_items.csv (send to the judge: no system names, shuffled) and judge_key.csv (keep private).
    A share of items (dup_frac) is repeated under new ids to measure judge self-consistency."""
    ref = pd.read_csv(f"{data_dir}/splits/{split}.csv")
    st = pd.read_csv(f"{data_dir}/structured/structured.csv")[["text_id", "part1", "part3"]]
    ref = ref.merge(st, on="text_id", validate="many_to_one")
    rows = []
    for name, path in preds.items():
        p = pd.read_csv(path, keep_default_na=False)[["text_id", "direction", "prediction"]]
        m = ref.merge(p, on=["text_id", "direction"], how="left", validate="one_to_one")
        miss = m.prediction.isna().sum()
        assert miss == 0, f"{name}: {miss} test items have no prediction"
        m["system"] = name
        rows.append(m)
    allp = pd.concat(rows, ignore_index=True)

    rng = np.random.default_rng(seed)
    allp["uid"] = range(len(allp))
    allp["dup_uid"] = -1
    n_dup = int(round(dup_frac * len(allp)))
    if n_dup:
        dup = allp.iloc[rng.choice(len(allp), n_dup, replace=False)].copy()
        dup["dup_uid"] = dup["uid"]
        dup["uid"] = range(len(allp), len(allp) + n_dup)
        allp = pd.concat([allp, dup], ignore_index=True)
    allp = allp.iloc[rng.permutation(len(allp))].reset_index(drop=True)
    allp["item_id"] = [f"J{i + 1:04d}" for i in range(len(allp))]
    uid2item = dict(zip(allp.uid, allp.item_id))
    allp["dup_of"] = allp.dup_uid.map(lambda u: uid2item[u] if u >= 0 else "")

    os.makedirs(out_dir, exist_ok=True)
    allp[["item_id", "direction", "part1", "part2", "part3", "question", "answer", "prediction"]] \
        .rename(columns={"answer": "gold_answer", "prediction": "candidate_answer"}) \
        .to_csv(f"{out_dir}/judge_items.csv", index=False)
    allp[["item_id", "system", "text_id", "direction", "dup_of"]].to_csv(f"{out_dir}/judge_key.csv", index=False)
    print(f"{len(allp)} judge items ({len(allp) - n_dup} unique + {n_dup} hidden duplicates) "
          f"from {len(preds)} systems -> {out_dir}")
    print("send judge_items.csv only; keep judge_key.csv private")

In [ ]:
# ---- Turn judge scores into the final numbers ----
# judge_scores.csv columns: item_id, score (2 = correct, 1 = partial, 0 = wrong), reason (optional)
import json, re


def tr_lower(s):
    return s.replace("I", "ı").replace("İ", "i").lower()


def tokens(s):
    return re.findall(r"\w+", tr_lower(str(s)))


def rouge_l_f1(pred, ref):
    p, r = tokens(pred), tokens(ref)
    if not p or not r:
        return 0.0
    dp = [[0] * (len(r) + 1) for _ in range(len(p) + 1)]
    for i in range(len(p)):
        for j in range(len(r)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if p[i] == r[j] else max(dp[i][j + 1], dp[i + 1][j])
    lcs = dp[-1][-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)


def chrf(preds, refs):
    from sacrebleu.metrics import CHRF
    return CHRF().corpus_score(list(preds), [list(refs)]).score


def paired_bootstrap(a, b, n=10000, seed=42):
    """95% CI of mean(b) - mean(a) over paired items."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = b[idx].mean(1) - a[idx].mean(1)
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def mcnemar_exact(a_ok, b_ok):
    from scipy.stats import binomtest
    b01 = int(((~a_ok) & b_ok).sum())   # baseline wrong, compare right
    b10 = int((a_ok & ~b_ok).sum())     # baseline right, compare wrong
    p = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    return b01, b10, float(p)


def summarize(g):
    return {
        "n": int(len(g)),
        "judged_accuracy": float((g.score / 2).mean()),          # correct=1, partial=0.5
        "strict_accuracy": float((g.score == 2).mean()),         # correct only
        "lenient_accuracy": float((g.score >= 1).mean()),        # correct or partial
        "correct": int((g.score == 2).sum()), "partial": int((g.score == 1).sum()), "wrong": int((g.score == 0).sum()),
        "rougeL_f1": float(np.mean([rouge_l_f1(p, r) for p, r in zip(g.candidate_answer, g.gold_answer)])),
        "chrF": chrf(g.candidate_answer.astype(str), g.gold_answer.astype(str)),
    }


def score(judge_dir, scores_path, baseline="baseline", compare="finetuned"):
    """Writes report.md, report.json and per_item.csv in judge_dir."""
    items = pd.read_csv(f"{judge_dir}/judge_items.csv", keep_default_na=False)
    key = pd.read_csv(f"{judge_dir}/judge_key.csv", keep_default_na=False)
    sc = pd.read_csv(scores_path, keep_default_na=False)
    assert set(sc.item_id) == set(items.item_id), "scores must cover every judge item exactly once"
    assert sc.item_id.is_unique
    sc["score"] = sc.score.astype(int)
    assert sc.score.isin([0, 1, 2]).all(), "score must be 0, 1 or 2"
    df = items.merge(key, on=["item_id", "direction"]).merge(sc[["item_id", "score"]], on="item_id")

    # judge self-consistency on hidden duplicates
    dups = df[df.dup_of != ""]
    orig = df.set_index("item_id").score
    cons = {"n_duplicates": int(len(dups))}
    if len(dups):
        same = (dups.score.values == orig.loc[dups.dup_of].values)
        cons.update(exact_agreement=float(same.mean()),
                    keep_drop_agreement=float(((dups.score.values >= 1) == (orig.loc[dups.dup_of].values >= 1)).mean()))
    df = df[df.dup_of == ""].copy()

    report = {"judge_consistency": cons, "systems": {}}
    for name, g in df.groupby("system"):
        report["systems"][name] = {"overall": summarize(g), **{d: summarize(gg) for d, gg in g.groupby("direction")}}

    if baseline in report["systems"] and compare in report["systems"]:
        b = df[df.system == baseline].set_index(["text_id", "direction"]).sort_index()
        c = df[df.system == compare].set_index(["text_id", "direction"]).sort_index()
        assert b.index.equals(c.index), "systems were not judged on the same items"
        bs, cs = (b.score / 2).values, (c.score / 2).values
        lo, hi = paired_bootstrap(bs, cs)
        b01, b10, p = mcnemar_exact(b.score.values == 2, c.score.values == 2)
        base_acc, comp_acc = bs.mean(), cs.mean()
        trans = pd.crosstab(b.score.values, c.score.values).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
        report["comparison"] = {
            "baseline": baseline, "compare": compare,
            "judged_accuracy_baseline": float(base_acc), "judged_accuracy_compare": float(comp_acc),
            "delta_points": float(100 * (comp_acc - base_acc)),
            "delta_points_95ci": [100 * lo, 100 * hi],
            "relative_gain_percent": float(100 * (comp_acc - base_acc) / base_acc) if base_acc else None,
            "mcnemar_strict": {"baseline_wrong_compare_right": b01, "baseline_right_compare_wrong": b10, "p": p},
            "meets_10_points": bool(100 * (comp_acc - base_acc) >= 10),
            "transitions": {"rows_baseline_cols_compare_scores_0_1_2": trans.values.tolist()},
        }

    json.dump(report, open(f"{judge_dir}/report.json", "w"), indent=2, ensure_ascii=False)

    lines = ["# Judged evaluation", ""]
    lines += ["| system | split | n | judged acc | strict | lenient | ROUGE-L | chrF |", "|---|---|---|---|---|---|---|---|"]
    for name, s in report["systems"].items():
        for part, m in s.items():
            lines.append(f"| {name} | {part} | {m['n']} | {100*m['judged_accuracy']:.1f} | {100*m['strict_accuracy']:.1f} | "
                         f"{100*m['lenient_accuracy']:.1f} | {100*m['rougeL_f1']:.1f} | {m['chrF']:.1f} |")
    if "comparison" in report:
        c = report["comparison"]
        lines += ["", f"**{c['compare']} vs {c['baseline']}:** {c['delta_points']:+.1f} points judged accuracy "
                  f"(95% CI {c['delta_points_95ci'][0]:+.1f} to {c['delta_points_95ci'][1]:+.1f}), "
                  f"relative {c['relative_gain_percent']:+.1f}%. McNemar (strict) p = {c['mcnemar_strict']['p']:.4f}. "
                  f"Meets +10 points: {c['meets_10_points']}."]
    if cons["n_duplicates"]:
        lines += ["", f"Judge self-consistency on {cons['n_duplicates']} hidden duplicates: "
                  f"{100*cons['exact_agreement']:.1f}% same score, {100*cons['keep_drop_agreement']:.1f}% same pass/fail."]
    open(f"{judge_dir}/report.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("\n".join(lines))
    plot_results(report, f"{judge_dir}/results.png", baseline, compare)
    return report

In [ ]:
# ---- Results figure (also redraws from a saved report.json) ----
def plot_results(report, path, baseline="baseline", compare="finetuned"):
    """Three panels: judged accuracy by direction, score distribution, per-pair change."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    S = report["systems"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=150, gridspec_kw={"width_ratios": [1.2, 1, 0.9]})

    # (a) judged accuracy, grouped by direction
    ax = axes[0]
    parts, w = ["overall", "past", "future"], 0.36
    x = np.arange(len(parts))
    for k, (name, col) in enumerate([(baseline, C_BASE), (compare, C_FT)]):
        vals = [100 * S[name][p]["judged_accuracy"] for p in parts]
        bars = ax.bar(x + (k - 0.5) * (w + 0.02), vals, w, color=col, label=name)
        ax.bar_label(bars, fmt="%.1f", fontsize=8, color=C_TEXT, padding=2)
    ax.set_xticks(x, parts)
    top = max(100 * S[n][p]["judged_accuracy"] for n in (baseline, compare) for p in parts)
    ax.set_ylim(0, top * 1.3)                     # headroom so the legend never covers a bar label
    ax.set_ylabel("judged accuracy (%)", color=C_MUTED, fontsize=9)
    ax.set_title("Judged accuracy by question direction", loc="left", fontsize=11, color=C_TEXT)
    ax.legend(frameon=False, fontsize=9, ncol=2, loc="upper left")
    style_axes(ax)

    # (b) share of wrong / partial / correct per system (ordered levels: one blue ramp)
    ax = axes[1]
    levels = [("wrong", "#cde2fb"), ("partial", "#6da7ec"), ("correct", "#1c5cab")]
    names = [baseline, compare]
    left = np.zeros(2)
    for lvl, col in levels:
        v = np.array([100 * S[n]["overall"][lvl] / S[n]["overall"]["n"] for n in names])
        ax.barh(names, v, left=left, color=col, edgecolor="white", linewidth=2, label=lvl, height=0.55)
        for i, (l, vi) in enumerate(zip(left, v)):
            if vi >= 6:
                ax.text(l + vi / 2, i, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                        color=C_TEXT if lvl == "wrong" else "white")
        left += v
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("share of test pairs (%)", color=C_MUTED, fontsize=9)
    ax.set_title("Score distribution", loc="left", fontsize=11, color=C_TEXT)
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    style_axes(ax)
    ax.grid(False)

    # (c) how each test pair moved: baseline score (rows) -> compare score (columns)
    ax = axes[2]
    c = report.get("comparison")
    if c and "transitions" in c:
        m = np.array(c["transitions"]["rows_baseline_cols_compare_scores_0_1_2"])
        ax.imshow(m, cmap=LinearSegmentedColormap.from_list("blue", ["#f0f6fd", "#104281"]))
        for i in range(3):
            for j in range(3):
                ax.text(j, i, str(m[i, j]), ha="center", va="center", fontsize=10,
                        color="white" if m[i, j] > m.max() / 2 else C_TEXT)
        lab = ["wrong", "partial", "correct"]
        ax.set_xticks(range(3), lab, fontsize=9, color=C_MUTED)
        ax.set_yticks(range(3), lab, fontsize=9, color=C_MUTED)
        ax.set_xlabel(f"{compare} score", color=C_MUTED, fontsize=9)
        ax.set_ylabel(f"{baseline} score", color=C_MUTED, fontsize=9)
        up, down = int(np.triu(m, 1).sum()), int(np.tril(m, -1).sum())
        ax.set_title(f"Per-pair change: {up} up, {down} down", loc="left", fontsize=11, color=C_TEXT)
        ax.spines[:].set_visible(False)
    else:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.show()

## Step 1: build the judge file

In [ ]:
RUN_ID = "PASTE_RUN_ID_HERE"        # folder name under runs/
JUDGE = f"{DRIVE_ROOT}/eval/judge_{RUN_ID}"
make_judge_file({"baseline": f"{DRIVE_ROOT}/eval/baseline_untrained/predictions_test.csv",
                 "finetuned": f"{DRIVE_ROOT}/runs/{RUN_ID}/predictions_test.csv"}, JUDGE)
print("send this file to the judge:", f"{JUDGE}/judge_items.csv")

## Step 2: after `judge_scores.csv` is in the folder

In [ ]:
RUN_ID = "PASTE_RUN_ID_HERE"        # same as Step 1 (repeated so Step 2 can run on its own)
JUDGE = f"{DRIVE_ROOT}/eval/judge_{RUN_ID}"
report = score(JUDGE, f"{JUDGE}/judge_scores.csv")

Already scored? Redraw the figure from the saved report without re-scoring:

In [ ]:
plot_results(json.load(open(f"{JUDGE}/report.json")), f"{JUDGE}/results.png")